In [1]:
import lightgbm as lgb
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

import sys
#sys.path.append("/work/home/acychkqsp6/zrm/MasterThesis/tools")
sys.path.append("../../tools")
import evaluation as eva
import prepare as prep


In [2]:
NumSpecies = 20
featgam = NumSpecies**2 - NumSpecies

# The folder storing datasets splited latter.
folder_path = "../../Databases/Ver1.9/ver1.9.6_upto20_v2"

def prepare_lgbm_features(Xg, XA, Xd, NumSpecies):

    # Extract off-diagonal interaction terms
    offdiag_mask = ~np.eye(NumSpecies, dtype=bool)
    XA_flat = XA[:, offdiag_mask]

    # Concatenate all input features
    X = np.concatenate(
        [Xg, XA_flat, Xd],
        axis=1
    )

    return X.astype(np.float32)

lgb_params = {
    "n_estimators": 3000,
    "learning_rate": 0.03,
    "num_leaves": 31,
    "max_depth": -1,
    "min_child_samples": 20,
    "subsample": 0.8,
    "subsample_freq": 1,
    "colsample_bytree": 0.8,
    "reg_lambda": 1.0,
    "random_state": 42,
    "n_jobs": -1,
    "verbosity": -1
}

def split_train_val(X, y, test_size=0.2, random_state=42):

    idx = np.arange(len(y))

    idx_tr, idx_val = train_test_split(
        idx,
        test_size=test_size,
        random_state=random_state
    )

    return (
        X[idx_tr],
        X[idx_val],
        y[idx_tr],
        y[idx_val]
    )


In [3]:
def train_lgbm_regression(X_tr, y_tr, X_val, y_val):

    model = lgb.LGBMRegressor(
        objective="regression",
        **lgb_params
    )

    model.fit(
        X_tr,
        y_tr.reshape(-1),
        eval_set=[
            (X_val, y_val.reshape(-1))
        ],
        eval_metric="l2",
        callbacks=[
            lgb.early_stopping(8),
            lgb.log_evaluation(100)
        ]
    )

    return model

def train_lgbm_multiclass(X_tr, y_tr, X_val, y_val, n_class):

    model = lgb.LGBMClassifier(
        objective="multiclass",
        num_class=n_class,
        **lgb_params
    )

    model.fit(
        X_tr,
        y_tr.reshape(-1).astype(int),
        eval_set=[
            (X_val, y_val.reshape(-1).astype(int))
        ],
        eval_metric="multi_logloss",
        callbacks=[
            lgb.early_stopping(8),
            lgb.log_evaluation(100)
        ]
    )

    return model

def train_lgbm_switching(X_tr, y_tr, X_val, y_val, active_species):

    models = []

    for i in range(active_species):

        print(f"Training species {i+1}/{active_species}")

        y_train_i = y_tr[:, i].astype(int)
        y_val_i = y_val[:, i].astype(int)

        # Handle species with only one class in the training set
        if len(np.unique(y_train_i)) < 2:

            constant_prob = float(y_train_i[0])
            models.append(constant_prob)

            continue

        model_i = lgb.LGBMClassifier(
            objective="binary",
            **lgb_params
        )

        model_i.fit(
            X_tr,
            y_train_i,
            eval_set=[
                (X_val, y_val_i)
            ],
            eval_metric="binary_logloss",
            callbacks=[
                lgb.early_stopping(8),
                lgb.log_evaluation(100)
            ]
        )

        models.append(model_i)

    return models

def predict_lgbm_switching(models, X, active_species):

    y_prob = np.zeros(
        (len(X), active_species),
        dtype=float
    )

    for i, model in enumerate(models):

        if isinstance(model, (float, int)):
            y_prob[:, i] = model

        else:
            class_idx = np.where(model.classes_ == 1)[0][0]

            y_prob[:, i] = model.predict_proba(X)[:, class_idx]

    return y_prob

def train_lgbm_binary(X_tr, y_tr, X_val, y_val):

    model = lgb.LGBMClassifier(
        objective="binary",
        **lgb_params
    )

    model.fit(
        X_tr,
        y_tr.reshape(-1).astype(int),
        eval_set=[
            (X_val, y_val.reshape(-1).astype(int))
        ],
        eval_metric="binary_logloss",
        callbacks=[
            lgb.early_stopping(8),
            lgb.log_evaluation(100)
        ]
    )

    return model


In [5]:
task_num = 2 # 0: num. of stable states, 1: stable-state entropy, 2: diversity, 3: abundance, 3+NumSpecies: switching species, 3+2*NumSpecies: oscillation

for rep in range(1,4):
    #split_df = pd.read_csv(f"/work/home/acychkqsp6/zrm/MasterThesis/Principle_design_SynCom/Fig2/split_MD_v1/split_MD_{rep}.csv")
    #split_df = pd.read_csv(f"/work/home/acychkqsp6/zrm/MasterThesis/Principle_design_SynCom/Fig2/split_density_descend/split_{rep}.csv")    
    split_df = pd.read_csv(f"../Fig2/split_ver1.9.6_upto20_v2/split_{rep}.csv")

    #Trainset
    narray_tr = prep.load_split_dataset(
        split_df,
        folder_path,
        richness_list=[10],
        #richness_list=list(range(3,21)),
        subset="tr"
    )

    #Testset
    narray_te = prep.load_split_dataset(
        split_df,
        folder_path,
        richness_list=[10],
        #richness_list=list(range(3,21)),
        subset="te"
    )

    #-------------------------------------------------------
    Xg_tr, XA_tr, Xd_tr, y_tr, scaler_g, scaler_d, idx_tr = prep.prepare_train(
        narray_tr,
        NumSpecies,
        featgam,
        task_num,
        #sample_size=97000,
        random_state=42
    )

    Xg_te, XA_te, Xd_te, y_te, idx_te = prep.prepare_test(
        narray_te,
        NumSpecies,
        featgam,
        scaler_g,
        scaler_d,
        task_num,
        #sample_size=20000,
        random_state=42
    )

    X_tr = prepare_lgbm_features(Xg_tr, XA_tr, Xd_tr, NumSpecies)
    X_te = prepare_lgbm_features(Xg_te, XA_te, Xd_te, NumSpecies)

    X_tr, X_val, y_tr_split, y_val = split_train_val(X_tr, y_tr)

    model = train_lgbm_regression(
        X_tr, y_tr_split,
        X_val, y_val
    )

    y_pred = model.predict(X_te)
    metrics = eva.eval_regression(y_te.reshape(-1), y_pred)

    display(pd.DataFrame([metrics]))


tr shape = (78594, 465)
te shape = (19648, 465)
Training until validation scores don't improve for 8 rounds
[100]	valid_0's l2: 0.0146747
[200]	valid_0's l2: 0.0141985
Early stopping, best iteration is:
[216]	valid_0's l2: 0.0141893


,R2,MAE,MedianAE,RMSE,NRMSE,SpearmanR,PearsonR
0,0.729897,0.094529,0.082368,0.119402,0.519715,0.611746,0.854426


tr shape = (78594, 465)
te shape = (19648, 465)
Training until validation scores don't improve for 8 rounds
[100]	valid_0's l2: 0.0145965
[200]	valid_0's l2: 0.0141042
[300]	valid_0's l2: 0.0140496
Early stopping, best iteration is:
[330]	valid_0's l2: 0.0140365


,R2,MAE,MedianAE,RMSE,NRMSE,SpearmanR,PearsonR
0,0.71584,0.095402,0.082912,0.120539,0.533067,0.601977,0.846109


tr shape = (78594, 465)
te shape = (19648, 465)
Training until validation scores don't improve for 8 rounds
[100]	valid_0's l2: 0.0147133
[200]	valid_0's l2: 0.0141884
Early stopping, best iteration is:
[242]	valid_0's l2: 0.0141576


,R2,MAE,MedianAE,RMSE,NRMSE,SpearmanR,PearsonR
0,0.71325,0.095689,0.082851,0.121326,0.53549,0.59573,0.844595


In [7]:
task_num = 0 # 0: num. of stable states, 1: stable-state entropy, 2: diversity, 3: abundance, 3+NumSpecies: switching species, 3+2*NumSpecies: oscillation

for rep in range(1,4):
    #split_df = pd.read_csv(f"/work/home/acychkqsp6/zrm/MasterThesis/Principle_design_SynCom/Fig2/split_MD_v1/split_MD_{rep}.csv")
    #split_df = pd.read_csv(f"/work/home/acychkqsp6/zrm/MasterThesis/Principle_design_SynCom/Fig2/split_density_descend/split_{rep}.csv")    
    split_df = pd.read_csv(f"../Fig2/split_ver1.9.6_upto20_v2/split_{rep}.csv")

    #Trainset
    narray_tr = prep.load_split_dataset(
        split_df,
        folder_path,
        richness_list=[10],
        #richness_list=list(range(3,21)),
        subset="tr"
    )

    #Testset
    narray_te = prep.load_split_dataset(
        split_df,
        folder_path,
        richness_list=[10],
        #richness_list=list(range(3,21)),
        subset="te"
    )

    #-------------------------------------------------------
    Xg_tr, XA_tr, Xd_tr, y_tr, scaler_g, scaler_d, idx_tr = prep.prepare_train(
        narray_tr,
        NumSpecies,
        featgam,
        task_num,
        #sample_size=97000,
        random_state=42
    )

    Xg_te, XA_te, Xd_te, y_te, idx_te = prep.prepare_test(
        narray_te,
        NumSpecies,
        featgam,
        scaler_g,
        scaler_d,
        task_num,
        #sample_size=20000,
        random_state=42
    )

    X_tr = prepare_lgbm_features(Xg_tr, XA_tr, Xd_tr, NumSpecies)
    X_te = prepare_lgbm_features(Xg_te, XA_te, Xd_te, NumSpecies)

    X_tr, X_val, y_tr_split, y_val = split_train_val(X_tr, y_tr)

    model = train_lgbm_multiclass(
        X_tr,
        y_tr_split,
        X_val,
        y_val,
        n_class=NumSpecies
    )

    y_prob_raw = model.predict_proba(X_te)

    y_prob = np.zeros(
        (len(X_te), NumSpecies),
        dtype=float
    )

    y_prob[:, model.classes_.astype(int)] = y_prob_raw

    metrics = eva.eval_multiclass(
        NumSpecies,
        y_te.reshape(-1),
        y_prob
    )

    display(pd.DataFrame([metrics]))


tr shape = (78594, 465)
te shape = (19648, 465)
Training until validation scores don't improve for 8 rounds
[100]	valid_0's multi_logloss: 1.24048
[200]	valid_0's multi_logloss: 1.22227
Early stopping, best iteration is:
[280]	valid_0's multi_logloss: 1.22047


,Accuracy,Balanced_Acc,F1_macro,F1_weighted,class_dist,Top2_Acc,MAE_ord_expect,RMSE_ord_expect,R2_ord_expect,MAE_ord_argmax,RMSE_ord_argmax
0,0.465086,0.253503,0.259789,0.452206,"[0.3597821661237785, 0.2588558631921824, 0.178...",0.724908,0.721763,0.975922,0.652299,0.705619,1.061044


tr shape = (78594, 465)
te shape = (19648, 465)
Training until validation scores don't improve for 8 rounds
[100]	valid_0's multi_logloss: 1.23805
[200]	valid_0's multi_logloss: 1.21958
Early stopping, best iteration is:
[205]	valid_0's multi_logloss: 1.21927


,Accuracy,Balanced_Acc,F1_macro,F1_weighted,class_dist,Top2_Acc,MAE_ord_expect,RMSE_ord_expect,R2_ord_expect,MAE_ord_argmax,RMSE_ord_argmax
0,0.464118,0.248914,0.253288,0.449948,"[0.35962947882736157, 0.263182003257329, 0.177...",0.724959,0.723409,0.97744,0.650895,0.70343,1.057416


tr shape = (78594, 465)
te shape = (19648, 465)
Training until validation scores don't improve for 8 rounds
[100]	valid_0's multi_logloss: 1.23753
[200]	valid_0's multi_logloss: 1.21994
Early stopping, best iteration is:
[232]	valid_0's multi_logloss: 1.21893


,Accuracy,Balanced_Acc,F1_macro,F1_weighted,class_dist,Top2_Acc,MAE_ord_expect,RMSE_ord_expect,R2_ord_expect,MAE_ord_argmax,RMSE_ord_argmax
0,0.467274,0.253664,0.265107,0.454027,"[0.35479438110749184, 0.26715187296416937, 0.1...",0.728573,0.727465,0.987811,0.638366,0.705619,1.068786


In [8]:
task_num = 3+NumSpecies # 0: num. of stable states, 1: stable-state entropy, 2: diversity, 3: abundance, 3+NumSpecies: switching species, 3+2*NumSpecies: oscillation

for rep in range(1,4):
    #split_df = pd.read_csv(f"/work/home/acychkqsp6/zrm/MasterThesis/Principle_design_SynCom/Fig2/split_MD_v1/split_MD_{rep}.csv")
    #split_df = pd.read_csv(f"/work/home/acychkqsp6/zrm/MasterThesis/Principle_design_SynCom/Fig2/split_density_descend/split_{rep}.csv")    
    split_df = pd.read_csv(f"../Fig2/split_ver1.9.6_upto20_v2/split_{rep}.csv")

    #Trainset
    narray_tr = prep.load_split_dataset(
        split_df,
        folder_path,
        richness_list=[10],
        #richness_list=list(range(3,21)),
        subset="tr"
    )

    #Testset
    narray_te = prep.load_split_dataset(
        split_df,
        folder_path,
        richness_list=[10],
        #richness_list=list(range(3,21)),
        subset="te"
    )

    #-------------------------------------------------------
    Xg_tr, XA_tr, Xd_tr, y_tr, scaler_g, scaler_d, idx_tr = prep.prepare_train(
        narray_tr,
        NumSpecies,
        featgam,
        task_num,
        #sample_size=97000,
        random_state=42
    )

    Xg_te, XA_te, Xd_te, y_te, idx_te = prep.prepare_test(
        narray_te,
        NumSpecies,
        featgam,
        scaler_g,
        scaler_d,
        task_num,
        #sample_size=20000,
        random_state=42
    )

    X_tr = prepare_lgbm_features(Xg_tr, XA_tr, Xd_tr, NumSpecies)
    X_te = prepare_lgbm_features(Xg_te, XA_te, Xd_te, NumSpecies)

    X_tr, X_val, y_tr_split, y_val = split_train_val(X_tr, y_tr)

    rich = 10

    models = train_lgbm_switching(
        X_tr,
        y_tr_split,
        X_val,
        y_val,
        active_species=rich
    )

    y_prob = predict_lgbm_switching(
        models,
        X_te,
        active_species=rich
    )

    threshold = 0.3
    metrics_list = []

    for i in range(rich):

        metrics_i = eva.eval_binary(
            y_te[:, i],
            y_prob[:, i],
            threshold=threshold
        )

        metrics_i["Species"] = i
        metrics_list.append(metrics_i)

    df_metrics_species = pd.DataFrame(metrics_list)

    metrics = (
        df_metrics_species
        .drop(columns=["Species"])
        .mean()
        .to_dict()
    )

    metrics["test_richness"] = rich

    display(pd.DataFrame([metrics]))


tr shape = (78594, 465)
te shape = (19648, 465)
Training species 1/10
Training until validation scores don't improve for 8 rounds
[100]	valid_0's binary_logloss: 0.430562
[200]	valid_0's binary_logloss: 0.404844
[300]	valid_0's binary_logloss: 0.393581
[400]	valid_0's binary_logloss: 0.388307
Early stopping, best iteration is:
[461]	valid_0's binary_logloss: 0.386457
Training species 2/10
Training until validation scores don't improve for 8 rounds
[100]	valid_0's binary_logloss: 0.427535
[200]	valid_0's binary_logloss: 0.401039
[300]	valid_0's binary_logloss: 0.389428
[400]	valid_0's binary_logloss: 0.383978
[500]	valid_0's binary_logloss: 0.381011
Early stopping, best iteration is:
[556]	valid_0's binary_logloss: 0.379961
Training species 3/10
Training until validation scores don't improve for 8 rounds
[100]	valid_0's binary_logloss: 0.428589
[200]	valid_0's binary_logloss: 0.402152
[300]	valid_0's binary_logloss: 0.391525
[400]	valid_0's binary_logloss: 0.38557
Early stopping, best i

,Accuracy,Balanced_Acc,F1,MCC,AUC,AUPRC,AUPRC_rel,Brier,pos_rate,test_richness
0,0.805553,0.81329,0.722616,0.589107,0.897327,0.800702,1.633387,0.119064,0.304087,10


tr shape = (78594, 465)
te shape = (19648, 465)
Training species 1/10
Training until validation scores don't improve for 8 rounds
[100]	valid_0's binary_logloss: 0.42341
[200]	valid_0's binary_logloss: 0.39741
[300]	valid_0's binary_logloss: 0.3866
[400]	valid_0's binary_logloss: 0.38105
[500]	valid_0's binary_logloss: 0.378161
Early stopping, best iteration is:
[522]	valid_0's binary_logloss: 0.377801
Training species 2/10
Training until validation scores don't improve for 8 rounds
[100]	valid_0's binary_logloss: 0.424524
[200]	valid_0's binary_logloss: 0.398876
[300]	valid_0's binary_logloss: 0.38794
[400]	valid_0's binary_logloss: 0.382824
[500]	valid_0's binary_logloss: 0.380191
Early stopping, best iteration is:
[590]	valid_0's binary_logloss: 0.378082
Training species 3/10
Training until validation scores don't improve for 8 rounds
[100]	valid_0's binary_logloss: 0.428031
[200]	valid_0's binary_logloss: 0.401846
[300]	valid_0's binary_logloss: 0.390424
[400]	valid_0's binary_logl

,Accuracy,Balanced_Acc,F1,MCC,AUC,AUPRC,AUPRC_rel,Brier,pos_rate,test_richness
0,0.801445,0.810257,0.719423,0.583021,0.894873,0.795958,1.604622,0.120916,0.305604,10


tr shape = (78594, 465)
te shape = (19648, 465)
Training species 1/10
Training until validation scores don't improve for 8 rounds
[100]	valid_0's binary_logloss: 0.427498
[200]	valid_0's binary_logloss: 0.40169
[300]	valid_0's binary_logloss: 0.390436
[400]	valid_0's binary_logloss: 0.384786
[500]	valid_0's binary_logloss: 0.381358
Early stopping, best iteration is:
[534]	valid_0's binary_logloss: 0.380838
Training species 2/10
Training until validation scores don't improve for 8 rounds
[100]	valid_0's binary_logloss: 0.426865
[200]	valid_0's binary_logloss: 0.400583
[300]	valid_0's binary_logloss: 0.390236
[400]	valid_0's binary_logloss: 0.385165
[500]	valid_0's binary_logloss: 0.382832
Early stopping, best iteration is:
[539]	valid_0's binary_logloss: 0.38211
Training species 3/10
Training until validation scores don't improve for 8 rounds
[100]	valid_0's binary_logloss: 0.422537
[200]	valid_0's binary_logloss: 0.396165
[300]	valid_0's binary_logloss: 0.385085
[400]	valid_0's binary_

,Accuracy,Balanced_Acc,F1,MCC,AUC,AUPRC,AUPRC_rel,Brier,pos_rate,test_richness
0,0.801374,0.806607,0.717169,0.578568,0.892454,0.794026,1.586051,0.122044,0.307054,10


In [5]:
task_num = 3+2*NumSpecies # 0: num. of stable states, 1: stable-state entropy, 2: diversity, 3: abundance, 3+NumSpecies: switching species, 3+2*NumSpecies: oscillation

folder_path = "../../Databases/Ver1.9/ver1.9.6_upto20_fl_v2"

for rep in range(1,4):

    split_df = pd.read_csv(f"../Fig2/split_ver1.9.6_upto20_fl_v2/split_{rep}.csv")

    #Trainset
    narray_tr, _ = prep.load_split_dataset2(
        split_df,
        folder_path,
        richness_list=[10],
        #richness_list=list(range(3,21)),
        subset="tr",
        sur_frac=False,
        MaxNum=NumSpecies,
    )

    #Testset
    narray_te, _ = prep.load_split_dataset2(
        split_df,
        folder_path,
        richness_list=[10],
        #richness_list=list(range(3,21)),
        subset="te",
        sur_frac=False,
        MaxNum=NumSpecies,
    )

    #-------------------------------------------------------
    Xg_tr, XA_tr, Xd_tr, y_tr, scaler_g, scaler_d, idx_tr = prep.prepare_train(
        narray_tr,
        NumSpecies,
        featgam,
        task_num,
        #sample_size=97000,
        random_state=42
    )

    Xg_te, XA_te, Xd_te, y_te, idx_te = prep.prepare_test(
        narray_te,
        NumSpecies,
        featgam,
        scaler_g,
        scaler_d,
        task_num,
        #sample_size=20000,
        random_state=42
    )

    X_tr = prepare_lgbm_features(Xg_tr, XA_tr, Xd_tr, NumSpecies)
    X_te = prepare_lgbm_features(Xg_te, XA_te, Xd_te, NumSpecies)

    X_tr, X_val, y_tr_split, y_val = split_train_val(X_tr, y_tr)

    model = train_lgbm_binary(
        X_tr,
        y_tr_split,
        X_val,
        y_val
    )

    y_prob = model.predict_proba(X_te)[:, 1]

    threshold = 0.8

    metrics = eva.eval_binary(
        y_te.reshape(-1),
        y_prob,
        threshold=threshold
    )

    display(pd.DataFrame([metrics]))


tr shape = (113980, 465)
te shape = (28494, 465)
Training until validation scores don't improve for 8 rounds
[100]	valid_0's binary_logloss: 0.452177
[200]	valid_0's binary_logloss: 0.416293
[300]	valid_0's binary_logloss: 0.404148
[400]	valid_0's binary_logloss: 0.395785
[500]	valid_0's binary_logloss: 0.389341
[600]	valid_0's binary_logloss: 0.38368
[700]	valid_0's binary_logloss: 0.378804
[800]	valid_0's binary_logloss: 0.374311
[900]	valid_0's binary_logloss: 0.370197
[1000]	valid_0's binary_logloss: 0.366563
[1100]	valid_0's binary_logloss: 0.36308
[1200]	valid_0's binary_logloss: 0.359917
[1300]	valid_0's binary_logloss: 0.357034
[1400]	valid_0's binary_logloss: 0.354379
[1500]	valid_0's binary_logloss: 0.352071
[1600]	valid_0's binary_logloss: 0.349721
[1700]	valid_0's binary_logloss: 0.347774
[1800]	valid_0's binary_logloss: 0.345901
[1900]	valid_0's binary_logloss: 0.344198
[2000]	valid_0's binary_logloss: 0.342464
[2100]	valid_0's binary_logloss: 0.34088
[2200]	valid_0's bina

,Accuracy,Balanced_Acc,F1,MCC,AUC,AUPRC,AUPRC_rel,Brier,pos_rate
0,0.836843,0.773295,0.706558,0.642735,0.914471,0.873713,1.504837,0.097934,0.34881


tr shape = (113980, 465)
te shape = (28494, 465)
Training until validation scores don't improve for 8 rounds
[100]	valid_0's binary_logloss: 0.450487
[200]	valid_0's binary_logloss: 0.413484
[300]	valid_0's binary_logloss: 0.399985
[400]	valid_0's binary_logloss: 0.391274
[500]	valid_0's binary_logloss: 0.38515
[600]	valid_0's binary_logloss: 0.379623
[700]	valid_0's binary_logloss: 0.374417
[800]	valid_0's binary_logloss: 0.36992
[900]	valid_0's binary_logloss: 0.36589
[1000]	valid_0's binary_logloss: 0.36245
[1100]	valid_0's binary_logloss: 0.359025
[1200]	valid_0's binary_logloss: 0.356018
[1300]	valid_0's binary_logloss: 0.353259
[1400]	valid_0's binary_logloss: 0.35052
[1500]	valid_0's binary_logloss: 0.348052
[1600]	valid_0's binary_logloss: 0.345746
[1700]	valid_0's binary_logloss: 0.343701
[1800]	valid_0's binary_logloss: 0.34199
[1900]	valid_0's binary_logloss: 0.340025
[2000]	valid_0's binary_logloss: 0.338265
[2100]	valid_0's binary_logloss: 0.336801
[2200]	valid_0's binary_

,Accuracy,Balanced_Acc,F1,MCC,AUC,AUPRC,AUPRC_rel,Brier,pos_rate
0,0.838457,0.775048,0.709241,0.644294,0.914682,0.874133,1.519985,0.09696,0.34688


tr shape = (113980, 465)
te shape = (28494, 465)
Training until validation scores don't improve for 8 rounds
[100]	valid_0's binary_logloss: 0.449232
[200]	valid_0's binary_logloss: 0.41216
[300]	valid_0's binary_logloss: 0.398741
[400]	valid_0's binary_logloss: 0.390805
[500]	valid_0's binary_logloss: 0.384202
[600]	valid_0's binary_logloss: 0.378451
[700]	valid_0's binary_logloss: 0.37349
[800]	valid_0's binary_logloss: 0.369241
[900]	valid_0's binary_logloss: 0.3652
[1000]	valid_0's binary_logloss: 0.361478
[1100]	valid_0's binary_logloss: 0.358248
[1200]	valid_0's binary_logloss: 0.354967
[1300]	valid_0's binary_logloss: 0.3524
[1400]	valid_0's binary_logloss: 0.349903
[1500]	valid_0's binary_logloss: 0.34741
[1600]	valid_0's binary_logloss: 0.345302
[1700]	valid_0's binary_logloss: 0.343344
[1800]	valid_0's binary_logloss: 0.341579
[1900]	valid_0's binary_logloss: 0.339966
[2000]	valid_0's binary_logloss: 0.33804
[2100]	valid_0's binary_logloss: 0.336467
[2200]	valid_0's binary_lo

,Accuracy,Balanced_Acc,F1,MCC,AUC,AUPRC,AUPRC_rel,Brier,pos_rate
0,0.827297,0.760509,0.685539,0.621545,0.910369,0.868093,1.483228,0.101723,0.349582
